# Radiology Knowledge Graph Extraction with CheXpert Plus (ReXKG)

This notebook walks through the full **ReXKG pipeline** on the CheXpert Plus dataset:

1. Load the CheXpert Plus dataset (`CheXpertPlusDataset`)
2. Apply the radiology KG extraction task (`RadiologyKGExtractionTask`)
3. **GPT-4o entity extraction** on a small subset (few-shot, mirrors `gpt4_entity_extraction.py`)
4. **GPT-4o relation extraction** on the entities from step 3 (mirrors `gpt4_relation_extraction.py`)
5. Initialize and run the `ReXKGModel` (BERT-based span NER + pairwise RE)
6. Build and inspect a structured Knowledge Graph

**Papers**
- ReXKG: https://arxiv.org/abs/2408.14397
- CheXpert Plus: https://arxiv.org/abs/2405.19111

**Authors:** Aaron Miller (aaronm6@illinois.edu), Kathryn Thompson (kyt3@illinois.edu), Pushpendra Tiwari (pkt3@illinois.edu)

## Step 0: Install dependencies

In [ ]:
%pip install pyhealth transformers ipywidgets openai tqdm networkx matplotlib

## Step 1: Load the CheXpert Plus Dataset

Download `df_chexpert_plus_240401.csv` from the [Stanford AIMI portal](https://stanfordaimi.azurewebsites.net/datasets/5158c524-d3ab-4e02-96e9-6ee9efc110a1) and set `DATA_ROOT` to the folder containing it.

Each row corresponds to one chest X-ray study. `path_to_image` is used as the patient identifier and `section_findings` is the text we extract knowledge from.

In [ ]:
from pyhealth.datasets import CheXpertPlusDataset

# Set this to the folder containing df_chexpert_plus_240401.csv
DATA_ROOT = "/path/to/chexpert_plus"

dataset = CheXpertPlusDataset(
    root=DATA_ROOT,
    dev=True,   # dev=True limits to 1 000 patients for quick exploration
)
dataset.stats()

## Step 2: Apply the Radiology KG Extraction Task

`RadiologyKGExtractionTask` converts each radiology report into a sample with:
- `text`: the findings text
- `entities`: (empty, populated by the model at inference)
- `relations`: (empty, populated by the model at inference)

In [ ]:
from pyhealth.tasks import RadiologyKGExtractionTask

task = RadiologyKGExtractionTask(
    findings_only=True,
    min_text_length=10,
)

samples = dataset.set_task(task)
print(f"Total samples: {len(samples)}")

## Step 3: GPT-4o Entity Extraction (small subset)

This section mirrors `gpt4_entity_extraction.py` from the original ReXKG pipeline.  
GPT-4o is prompted with a few-shot radiology example to extract clinical terms sentence-by-sentence.

Entity types: `anatomy`, `disorder_present`, `disorder_notpresent`, `procedures`,
`devices_present`, `devices_notpresent`, `size`, `concept`.

> **Cost note:** Each report costs ~\$0.01–0.03 with `gpt-4o`. Set `N_SAMPLES` small (e.g. 5–20) for exploration.

> **Security:** Do **not** hard-code your API key in this notebook. Use an environment variable or a secrets manager instead.

In [ ]:
import os

# ------------------------------------------------------------------ #
#  CONFIGURE: set your OpenAI credentials here OR export them as env  #
#  vars before launching the notebook (recommended).                  #
# ------------------------------------------------------------------ #
OPENAI_API_KEY  = os.getenv("OPENAI_API_KEY",  "<YOUR_OPENAI_API_KEY_HERE>")
OPENAI_API_BASE = os.getenv("OPENAI_API_BASE", "https://api.openai.com/v1")

# Number of reports to run through GPT (keep small to control cost)
N_SAMPLES = 5

# Where to cache GPT results so interrupted runs can be resumed
GPT_ENTITY_CACHE    = "gpt4_entities_subset.json"
GPT_RELATION_CACHE  = "gpt4_relations_subset.json"

In [ ]:
import json, time
import openai
from tqdm import tqdm

openai.api_key  = OPENAI_API_KEY
openai.api_base = OPENAI_API_BASE


# ── Few-shot system prompt (from gpt4_entity_extraction.py) ──────────────── #
_NER_FEWSHOT = [
    {
        "context": (
            "<Input> Unchanged position of the left upper extremity PICC line. "
            "Again seen are surgical clips projecting over the right hemithorax. "
            "Increased stranding opacities are noted in the left retrocardiac region.<\\Input>"
        ),
        "response": (
            "{'Unchanged position of the left upper extremity PICC line.':"
            "{'Unchanged':'concept','position':'concept','left':'concept','upper':'concept',"
            "'extremity':'anatomy','PICC line':'device_present'},"
            "'Again seen are surgical clips projecting over the right hemithorax.':"
            "{'surgical clips':'device_present','right':'concept','hemithorax':'anatomy'},"
            "'Increased stranding opacities are noted in the left retrocardiac region.':"
            "{'Increased':'concept','stranding':'concept','opacities':'disorder_present',"
            "'left':'concept','retrocardiac':'anatomy','region':'anatomy'}}"
        ),
    }
]

_NER_SYSTEM = (
    "You are a radiologist performing clinical term extraction from the FINDINGS and "
    "IMPRESSION sections in the radiology report. "
    "Here a clinical term can be in ['anatomy','disorder_present','disorder_notpresent',"
    "'procedures','devices','concept','devices_present','devices_notpresent','size']. "
    "'anatomy' refers to the anatomical body; "
    "'disorder_present' refers to findings or diseases that are present; "
    "'disorder_notpresent' refers to findings or diseases that are NOT present; "
    "'procedures' refers to diagnostic/therapeutic procedures; "
    "'devices' refers to any medical instrument or apparatus; "
    "'size' refers to measurements, e.g. '3mm','4x5 cm'; "
    "'concept' refers to descriptors, modifiers, or severity qualifiers. "
    "Please extract terms one word at a time whenever possible. "
    "Note that terms like 'no' and 'no evidence of' are NOT entities. "
    "Given radiology text as <Input><sentence><sentence><\\Input>, "
    "reply with JSON: {'<sentence>':{'entity':'entity_type',...},...}"
)


def _ner_messages(text: str) -> list:
    msgs = [{"role": "system", "content": _NER_SYSTEM}]
    for s in _NER_FEWSHOT:
        msgs.append({"role": "user",      "content": s["context"]})
        msgs.append({"role": "assistant", "content": s["response"]})
    msgs.append({"role": "user", "content": f"<Input> {text} <\\Input>"})
    return msgs


def _estimate_cost(prompt_tokens: int, completion_tokens: int) -> float:
    return prompt_tokens * 0.005 / 1000 + completion_tokens * 0.015 / 1000


def gpt_extract_entities(text: str) -> tuple:
    """Return (parsed_dict_or_str, cost_usd)."""
    resp = openai.ChatCompletion.create(
        model="gpt-4o-2024-05-13",
        messages=_ner_messages(text),
        response_format={"type": "json_object"},
    )
    raw  = resp["choices"][0]["message"]["content"]
    cost = _estimate_cost(resp["usage"]["prompt_tokens"], resp["usage"]["completion_tokens"])
    try:
        return json.loads(raw), cost
    except Exception:
        return raw, cost


print("GPT entity-extraction helpers ready.")

In [ ]:
import pandas as pd

# Load a small slice of the CSV directly for GPT processing
df_gpt = pd.read_csv(f"{DATA_ROOT}/df_chexpert_plus_240401.csv").dropna(subset=["section_findings"]).head(N_SAMPLES)

# Resume from cache if it exists
try:
    with open(GPT_ENTITY_CACHE) as fh:
        entity_cache = json.load(fh)
    print(f"Loaded {len(entity_cache)} cached entity results.")
except FileNotFoundError:
    entity_cache = {}

total_cost = 0.0

for _, row in tqdm(df_gpt.iterrows(), total=len(df_gpt), desc="GPT entity extraction"):
    pid  = row["path_to_image"]
    text = row["section_findings"]
    if pid in entity_cache:
        continue
    try:
        res, cost = gpt_extract_entities(text)
        total_cost += cost
        entity_cache[pid] = {"section_findings": text, "res": res, "cost": cost}
    except Exception as exc:
        print(f"  Error on {pid}: {exc}")
        time.sleep(1)
    with open(GPT_ENTITY_CACHE, "w") as fh:
        json.dump(entity_cache, fh, indent=2)

print(f"\nDone. Cumulative cost: ${total_cost:.4f}")
print(f"Results saved to {GPT_ENTITY_CACHE}")

In [ ]:
# Inspect entity extraction results for the first study
first_key = next(iter(entity_cache))
print("Patient/Study:", first_key)
print("Findings snippet:", entity_cache[first_key]["section_findings"][:200])
print("\nExtracted entities (per sentence):")
for sent, ents in entity_cache[first_key]["res"].items():
    print(f"  [{sent[:60]}...]")
    for term, etype in ents.items():
        print(f"    {term!r:30s} -> {etype}")

## Step 4: GPT-4o Relation Extraction

Takes the entity dict from Step 3 as input and extracts directed relations between entities.

Relation types: `modify`, `located_at`, `suggestive_of`.

In [ ]:
# ── Few-shot system prompt (from gpt4_relation_extraction.py) ─────────────── #
_RE_FEWSHOT = [
    {
        "context": (
            "{'Bones are stable with mild degenerative changes of the spine.':"
            "{'Bones':'anatomy','stable':'concept','mild':'concept',"
            "'degenerative changes':'disorder_present','spine':'anatomy'}}"
        ),
        "response": (
            "{'Bones are stable with mild degenerative changes of the spine.':"
            "[{'stable':'Bones','relation':'modify'},"
            "{'mild':'degenerative changes','relation':'modify'},"
            "{'degenerative changes':'spine','relation':'located_at'}]}"
        ),
    },
    {
        "context": (
            "{'A dense retrocardiac opacity remains present with slight blunting of the "
            "left costophrenic angle, suggestive of a small effusion.':"
            "{'dense':'concept','retrocardiac':'anatomy','opacity':'disorder_present',"
            "'slight':'concept','blunting':'disorder_present','left':'concept',"
            "'costophrenic':'anatomy','angle':'anatomy','small':'concept','effusion':'disorder_present'}}"
        ),
        "response": (
            "{'A dense retrocardiac opacity remains present with slight blunting of the "
            "left costophrenic angle, suggestive of a small effusion.':"
            "[{'dense':'opacity','relation':'modify'},{'opacity':'retrocardiac','relation':'located_at'},"
            "{'slight':'blunting','relation':'modify'},{'blunting':'angle','relation':'modify'},"
            "{'left':'costophrenic','relation':'modify'},{'small':'effusion','relation':'modify'},"
            "{'effusion':'costophrenic','relation':'located_at'},"
            "{'opacity':'effusion','relation':'suggestive_of'},"
            "{'blunting':'effusion','relation':'suggestive_of'}]}"
        ),
    },
]

_RE_SYSTEM = (
    "You are a radiologist performing relation extraction of entities from radiology reports. "
    "Relations: 'modify' (source modifies target), 'located_at' (source located at target anatomy), "
    "'suggestive_of' (source finding suggests target disease). "
    "Every concept->anatomy link should use 'modify'. "
    "Input JSON format: {'sentence':{'entity':'entity_type',...},...} "
    "Reply JSON: {'sentence':[{'source_entity':'target_entity','relation':'relation'},...]}" 
)


def _re_messages(entity_json: str) -> list:
    msgs = [{"role": "system", "content": _RE_SYSTEM}]
    for s in _RE_FEWSHOT:
        msgs.append({"role": "user",      "content": s["context"]})
        msgs.append({"role": "assistant", "content": s["response"]})
    msgs.append({"role": "user", "content": entity_json})
    return msgs


def gpt_extract_relations(entity_dict: dict) -> tuple:
    """Return (parsed_dict_or_str, cost_usd)."""
    resp = openai.ChatCompletion.create(
        model="gpt-4o-2024-05-13",
        messages=_re_messages(json.dumps(entity_dict)),
        response_format={"type": "json_object"},
    )
    raw  = resp["choices"][0]["message"]["content"]
    cost = _estimate_cost(resp["usage"]["prompt_tokens"], resp["usage"]["completion_tokens"])
    try:
        return json.loads(raw), cost
    except Exception:
        return raw, cost


print("GPT relation-extraction helpers ready.")

In [ ]:
# Resume from cache if it exists
try:
    with open(GPT_RELATION_CACHE) as fh:
        relation_cache = json.load(fh)
    print(f"Loaded {len(relation_cache)} cached relation results.")
except FileNotFoundError:
    relation_cache = {}

total_re_cost = 0.0

for pid, entry in tqdm(entity_cache.items(), desc="GPT relation extraction"):
    if pid in relation_cache:
        continue
    entity_dict = entry.get("res", {})
    if not entity_dict:
        continue
    try:
        res, cost = gpt_extract_relations(entity_dict)
        total_re_cost += cost
        relation_cache[pid] = {**entry, "res_relation": res, "cost_relation": cost}
    except Exception as exc:
        print(f"  Error on {pid}: {exc}")
        time.sleep(1)
    with open(GPT_RELATION_CACHE, "w") as fh:
        json.dump(relation_cache, fh, indent=2)

print(f"\nDone. Cumulative relation-extraction cost: ${total_re_cost:.4f}")
print(f"Results saved to {GPT_RELATION_CACHE}")

In [ ]:
# Inspect relation extraction results for the first study
first_key = next(iter(relation_cache))
print("Patient/Study:", first_key)
print("\nExtracted relations (per sentence):")
for sent, rels in relation_cache[first_key].get("res_relation", {}).items():
    print(f"  [{sent[:60]}...]")
    if isinstance(rels, list):
        for r in rels:
            items = list(r.items())
            rel_type = r.get("relation", "?")
            entities  = [(k, v) for k, v in items if k != "relation"]
            if entities:
                src, tgt = entities[0]
                print(f"    '{src}' --[{rel_type}]--> '{tgt}'")

## Step 5: Initialize the ReXKG Model (BERT-based)

`ReXKGModel` attaches a **span NER head** and a **pairwise relation head** on top of a BERT encoder.  
Use this for large-scale inference after fine-tuning, or combine with the GPT outputs above for rapid prototyping.

In [ ]:
from pyhealth.models import ReXKGModel

model = ReXKGModel(
    dataset=samples,
    bert_model_name="bert-base-uncased",
    max_span_length=8,
)
print(model)

### (Optional) Load a pretrained ReXKG checkpoint

If you have a checkpoint fine-tuned on the ReXKG / MIMIC NER task, load the weights here.
The **GPT-extracted** entities/relations from Steps 3–4 can also be used directly to assemble a KG without fine-tuning.

In [ ]:
import torch

CHECKPOINT_PATH = None  # e.g. "/path/to/rexkg_checkpoint.pt"

if CHECKPOINT_PATH:
    state = torch.load(CHECKPOINT_PATH, map_location="cpu")
    model.load_state_dict(state)
    print("Checkpoint loaded.")
else:
    print("Running with randomly initialized NER/RE heads (base BERT encoder weights only).")
    print("Predictions will not be meaningful without fine-tuning.")

## Step 5b: Run BERT Inference — Entity & Relation Extraction

In [ ]:
# Use the same small subset that was sent to GPT for a fair comparison
reports     = [v["section_findings"] for v in list(entity_cache.values())]
patient_ids = list(entity_cache.keys())

bert_entities = model.predict_entities(reports, batch_size=8)
for pid, ents in zip(patient_ids, bert_entities):
    print(f"\n{pid}: {len(ents)} entities")
    for e in ents:
        print(f"  [{e['type']}] '{e['text']}'")

In [ ]:
bert_relations = model.predict_relations(reports, bert_entities, batch_size=8)
for pid, rels in zip(patient_ids, bert_relations):
    print(f"\n{pid}: {len(rels)} relations")
    for r in rels:
        print(f"  '{r['subject']['text']}' --[{r['relation']}]--> '{r['object']['text']}'")

## Step 6: Build and Inspect the Knowledge Graph

`build_kg()` accepts either the BERT model outputs or the GPT-extracted results and assembles a deduplicated KG with:
- `nodes`: entity nodes (id, text, type)
- `edges`: relation triples (subject_id, relation, object_id)
- `subgraphs`: per-study subgraph

Below we demonstrate with the **BERT model outputs**. Swap in the GPT results by constructing analogous `reports`/`entities`/`relations` lists from `relation_cache`.

In [ ]:
kg = model.build_kg(reports=reports, patient_ids=patient_ids)

print(f"KG nodes : {len(kg['nodes'])}")
print(f"KG edges : {len(kg['edges'])}")

print("\nNodes:")
for n in kg["nodes"]:
    print(f"  [{n['id']}] {n['text']}  ({n['type']})")

print("\nEdges:")
node_map = {n["id"]: n["text"] for n in kg["nodes"]}
for e in kg["edges"]:
    print(f"  '{node_map[e['subject_id']]}' --[{e['relation']}]--> '{node_map[e['object_id']]}'")

In [ ]:
model.save_kg(kg, "chexpert_plus_kg_demo.json")
print("KG saved to chexpert_plus_kg_demo.json")

## Step 7: (Optional) Visualize the Subgraph

In [ ]:
try:
    import networkx as nx
    import matplotlib.pyplot as plt

    G = nx.DiGraph()
    node_map = {n["id"]: n["text"] for n in kg["nodes"]}
    for e in kg["edges"]:
        G.add_edge(node_map[e["subject_id"]], node_map[e["object_id"]], label=e["relation"])

    pos = nx.spring_layout(G, seed=42)
    edge_labels = nx.get_edge_attributes(G, "label")

    plt.figure(figsize=(12, 7))
    nx.draw(G, pos, with_labels=True, node_size=2000, node_color="lightblue", font_size=9, arrows=True)
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)
    plt.title("ReXKG Knowledge Graph — CheXpert Plus Demo")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Install networkx and matplotlib: pip install networkx matplotlib")

## Summary

| Step | Component | What it does |
|------|-----------|-------------|
| 1 | `CheXpertPlusDataset` | Loads CheXpert Plus CSV; one patient per X-ray study |
| 2 | `RadiologyKGExtractionTask` | Converts findings text into structured samples |
| 3 | GPT-4o entity extraction | Few-shot NER on small subset; saves to JSON cache |
| 4 | GPT-4o relation extraction | Few-shot RE on entity dict; saves to JSON cache |
| 5 | `ReXKGModel` | BERT encoder + span NER head + pairwise RE head |
| 5b | `predict_entities / predict_relations` | BERT-based inference at scale |
| 6 | `build_kg` / `save_kg` | Assembles and serialises deduplicated KG |
| 7 | NetworkX | Optional subgraph visualisation |